# Portfolio Sophistication Report Demo

This notebook codifies the chart from the *Mas sofisticacion = mejor portfolio?* discussion: compare simple and sophisticated portfolio construction objectives side by side, then inspect whether extra complexity actually improves the portfolio.

The demo uses the same defensive stock universe used elsewhere in RiskOptima documentation. Returns are synthetic and seeded so the notebook runs without a market-data or article dependency.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from riskoptima.reporting import (
    available_sophistication_methods,
    build_portfolio_sophistication_report,
    plot_portfolio_sophistication_report,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Define the existing RiskOptima portfolio universe

This is the same defensive/dividend-oriented stock universe shown in the main README and portfolio optimization notebook. We keep the labels and starting weights visible because they make the comparison easier to explain in interviews.

In [ ]:
asset_data = [
    {"Asset": "MO", "Weight": 0.04, "Label": "Altria Group Inc.", "MarketCap": 110.0e9},
    {"Asset": "NWN", "Weight": 0.14, "Label": "Northwest Natural Gas", "MarketCap": 1.8e9},
    {"Asset": "BKH", "Weight": 0.01, "Label": "Black Hills Corp.", "MarketCap": 4.5e9},
    {"Asset": "ED", "Weight": 0.01, "Label": "Con Edison", "MarketCap": 30.0e9},
    {"Asset": "PEP", "Weight": 0.09, "Label": "PepsiCo Inc.", "MarketCap": 255.0e9},
    {"Asset": "NFG", "Weight": 0.16, "Label": "National Fuel Gas", "MarketCap": 5.6e9},
    {"Asset": "KO", "Weight": 0.06, "Label": "Coca-Cola Company", "MarketCap": 275.0e9},
    {"Asset": "FRT", "Weight": 0.28, "Label": "Federal Realty Inv. Trust", "MarketCap": 9.8e9},
    {"Asset": "GPC", "Weight": 0.16, "Label": "Genuine Parts Co.", "MarketCap": 25.3e9},
    {"Asset": "MSEX", "Weight": 0.05, "Label": "Middlesex Water Co.", "MarketCap": 2.4e9},
]
portfolio = pd.DataFrame(asset_data).set_index("Asset")
portfolio

## 2. Generate reproducible asset returns

The article question is about portfolio construction, not data downloading. We therefore create a realistic defensive-equity return panel with a shared market factor, sector-style factors, and asset-specific noise. This keeps the notebook deterministic and makes it safe for CI or offline execution.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.bdate_range("2014-04-08", "2024-06-28")
n_assets = len(portfolio)

market = rng.normal(0.00028, 0.0085, size=(len(dates), 1))
defensive = rng.normal(0.00008, 0.0040, size=(len(dates), 1))
real_assets = rng.normal(0.00005, 0.0060, size=(len(dates), 1))
idiosyncratic = rng.normal(0.0, 0.0065, size=(len(dates), n_assets))

betas = np.array([0.75, 0.55, 0.65, 0.50, 0.70, 0.80, 0.62, 1.00, 0.82, 0.45])
defensive_loadings = np.array([0.30, 0.75, 0.65, 0.80, 0.50, 0.45, 0.55, 0.15, 0.35, 0.70])
real_asset_loadings = np.array([0.10, 0.25, 0.20, 0.20, 0.05, 0.35, 0.05, 0.75, 0.10, 0.15])
drifts = np.array([0.00018, 0.00012, 0.00013, 0.00011, 0.00020, 0.00016, 0.00018, 0.00014, 0.00017, 0.00010])

returns = pd.DataFrame(
    market @ betas.reshape(1, -1)
    + defensive @ defensive_loadings.reshape(1, -1)
    + real_assets @ real_asset_loadings.reshape(1, -1)
    + idiosyncratic
    + drifts,
    index=dates,
    columns=portfolio.index,
)
returns.head()

## 3. Review the available construction methods

The method names match the chart: `MV`, `CVaR`, `EVaR`, `RLVaR`, `WR`, `CDaR`, `EDaR`, `RLDaR`, `MDD`, and `1N`. Some labels are implemented as transparent proxies so users can run the comparison without adding a specialised external optimiser dependency.

In [ ]:
available_sophistication_methods()

## 4. Build the report

The report returns a standard RiskOptima `RiskReport`. The key outputs are `weights`, `returns`, `wealth`, and `performance_table`, which can feed a notebook, dashboard, or exported image.

In [ ]:
report = build_portfolio_sophistication_report(
    returns,
    methods=("MV", "CVaR", "EVaR", "RLVaR", "WR", "CDaR", "EDaR", "RLDaR", "MDD", "1N"),
    confidence=0.95,
    initial_value=100.0,
)

report.metrics["weights"].round(4)

## 5. Plot the article-style comparison chart

The top panel compares cumulative wealth. The lower panel is intentionally table-first: it lets a reviewer check whether higher complexity improved the risk-adjusted profile or merely overfit a specific objective.

In [ ]:
fig = plot_portfolio_sophistication_report(
    report,
    title="Does More Sophistication Improve the Portfolio?",
    figsize=(16, 9),
    table_fontsize=7,
)
plt.show()

## 6. Inspect the performance table directly

For automated workflows, use the raw table rather than parsing the plotted image. Values are already aligned by construction method.

In [ ]:
report.metrics["performance_table"].round(4)

## 7. Practical interpretation

The useful interview point is not that the most complex label always wins. A solid quant workflow compares a simple benchmark (`1N`), a classic optimiser (`MV`), tail-risk objectives (`CVaR`, `EVaR`, `RLVaR`), and drawdown objectives (`CDaR`, `EDaR`, `RLDaR`, `MDD`) under the same data and metrics. That makes the sophistication trade-off explicit.